# Phase 0 — 01 · Run

Predict genes → annotate → export GFF3 → score → sanity checks.

Every stage shells out to the same script a terminal run uses, so the notebook and the command
line cannot drift apart. Kernel: **Python 3.12 (geneweaver)**.

**Pod timeout is 2 h.** Stage 2 is the long one (DIAMOND × 2 + PyHMMER over Pfam-A, tens of
minutes at 4 threads). It keeps CPU above the 20 % idle threshold, but don't start it and walk
away for hours — outputs land on `/mnt/primary` and survive, so a killed pod means re-running a
stage, not losing work.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
import pandas as pd

assert sys.version_info[:2] == (3, 12), "switch to the Python 3.12 (geneweaver) kernel"

REPO = Path("/mnt/primary/Geneweaver_v3")
BUNDLE = REPO / "phase0-bundle"; CLUSTER = BUNDLE / "cluster"
cfg = json.loads((CLUSTER / "paths.json").read_text())
DB = Path(cfg["databases"]); THREADS = cfg.get("threads", 4); CPUS = cfg.get("cpus", 4)
P, S = REPO / "Phase-0", REPO / "Structural_annotation"
os.environ["PATH"] = f"{cfg['env_prefix']}/bin:" + os.environ["PATH"]
os.environ["GENEWEAVER_DB"] = str(DB)

def sh(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out[-3000:] if len(out) > 3000 else out, flush=True)
    if r.returncode != 0:
        raise RuntimeError(f"exit {r.returncode}")
    return out

print("repo:", REPO, "| db:", DB, "| threads:", THREADS)
print(subprocess.run(["diamond","--version"], capture_output=True, text=True).stdout.strip())

## 1 · Predict genes, then annotate

Every module on branch `dev` is a library class with no `__main__` block — `GenePredictor` included — so none of them runs as a script. `run_pipeline.py` drives all five.

This is the long stage: Pyrodigal, then DIAMOND against both Swiss-Prot builds and PyHMMER against Pfam-A. Domain and GO evidence are computed once; homology and combination run once per leakage condition. Tens of minutes at 4 threads.

Add `"--skip-genes"` to reuse an existing `pyrodigal_results/` if you are re-running.

In [ ]:
sh([sys.executable, BUNDLE / "scripts" / "run_pipeline.py",
    "--repo", REPO, "--databases", DB,
    "--threads", THREADS, "--cpus", CPUS])

## 2 · Export GFF3

In [ ]:
GENOME = P / "reference/ncbi_dataset/data/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.fna"
for cond, dbf in [("swissprot", "uniprot_sprot.fasta.gz"),
                  ("swissprot_no_ecoli", "uniprot_sprot_no_ecoli.fasta.gz")]:
    sh([sys.executable, "export_pipeline_gff3.py",
        "--annotation", S / "annotation_results" / f"ecoli_{cond}_annotation.tsv",
        "--genes-gff",  S / "pyrodigal_results" / "ecoli_genes.gff",
        "--swissprot",  DB / dbf,
        "--genome",     GENOME,
        "--output",     P / "pipeline" / f"pipeline_{cond}.gff3"], cwd=P)

## 3 · Score against the frozen reference

4,290 curated genes. Fixed string rules only — no LLM is used as a judge.

In [ ]:
sh([sys.executable, "score_gff.py",
    "--pred", "bakta=bakta/result.gff3",
    "--pred", "pipeline_swissprot=pipeline/pipeline_swissprot.gff3",
    "--pred", "pipeline_no_ecoli=pipeline/pipeline_swissprot_no_ecoli.gff3",
    "--outdir", "results"], cwd=P)

## 4 · Sanity checks — self-score and 10-gene deletion

Both must pass, or the scorer itself is not trustworthy.

In [ ]:
sh([sys.executable, "sanity_checks.py"], cwd=P)

## 5 · Result vs the committed baseline

Any `CHANGED` is a finding to explain before adopting the new number.

In [ ]:
m = pd.read_csv(P / "results" / "metrics.tsv", sep="\t").set_index("tool")
show = ["recall","precision","start_exact","symbol_match","product_match",
        "agrees","wrong_function","hypothetical_share"]
display((m[show] * 100).round(2).T)

EXPECTED = {"bakta": 0.9074, "pipeline_swissprot": 0.8771, "pipeline_no_ecoli": 0.5165}
print("\nfunction agreement vs committed baseline:")
for tool, want in EXPECTED.items():
    got = m.loc[tool, "agrees"]
    print(f"  {tool:22s} {got*100:6.2f}%   (baseline {want*100:.2f}%)  "
          f"{'MATCH' if abs(got-want) < 5e-4 else 'CHANGED'}")

---
### The 44 % needs its caveat

`wrong_function` for the no-*E. coli* condition is **a lower bound set by how strict the string
rules are, not a count of real errors** — they cannot credit "TrkA-N domain-containing protein"
as compatible with the K⁺ channel Kch. The hand-check sheets in `Phase-0/results/` turn it into
a defensible number. Don't quote it without this.

### Optional — local Bakta comparator

```bash
bash phase0-bundle/cluster/run_bakta_local.sh
```

Writes to `Phase-0/bakta_local/` (never the frozen `Phase-0/bakta/`), prints the version block
for your provenance and the `detected IPSs` leakage count, then the command to score both Bakta
runs side by side.